# ShiroRVC

Turn one voice into another — speaking or singing.

**Before you start:** *Runtime → Change runtime type → **T4 GPU***.

1. Run **Setup** once. It takes a few minutes.
2. Run **Start** and open the `gradio.live` link it prints.

[Repository](https://github.com/ShiromiyaG/ShiroRVC) · [Report a problem](https://github.com/ShiromiyaG/ShiroRVC/issues)


## 1. Setup

Installs Python 3.12, the repository and its dependencies. Run once per session.


In [ ]:
#@title Setup
import os
import pathlib
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ShiromiyaG/ShiroRVC.git"
ROOT = pathlib.Path("/content/ShiroRVC")
PYTHON_VERSION = "3.12"

# Kept in step with run-install.sh. Installing requirements.txt on its own
# would pull the default (CPU) torch build and quietly waste the GPU, so the
# CUDA wheels come from PyTorch's own index first.
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu130"
TORCH_PINS = ["torch==2.13.0", "torchvision==0.28.0", "torchaudio==2.11.0"]


def run(command):
    """Run a command, echoing it, and stop the cell if it fails."""
    print("$", " ".join(str(part) for part in command), flush=True)
    subprocess.run([str(part) for part in command], check=True)


def report_gpu():
    smi = shutil.which("nvidia-smi")
    if not smi:
        print("No GPU attached. Runtime > Change runtime type > T4 GPU.")
        return
    result = subprocess.run(
        [smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    print("GPU:", result.stdout.strip() or "none detected")


report_gpu()

# uv installs and manages its own interpreters, so Python 3.12 can be had
# without touching the one this notebook's kernel runs on. Replacing that one --
# the apt / update-alternatives recipes -- breaks the kernel: google.colab lives
# in the system site-packages and does not follow the swap.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
uv = shutil.which("uv") or "uv"
run([uv, "python", "install", PYTHON_VERSION])

if not ROOT.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, ROOT])
os.chdir(ROOT)

run([uv, "venv", "--python", PYTHON_VERSION, "env"])
python = ROOT / "env" / "bin" / "python"

run([uv, "pip", "install", "--python", python, "-q", *TORCH_PINS,
     "--upgrade", "--index-url", TORCH_INDEX_URL])
run([uv, "pip", "install", "--python", python, "-q", "-r", "requirements.txt"])

# Pitch extractors, content embedders and the executables the pipeline shells
# out to. Every option defaults to true, so this fetches the full set.
run([python, "core.py", "prerequisites"])

print()
print("Setup finished. Run the next cell to start the interface.")


## 2. Start

Launches the interface and prints a public link. Keep this cell running while you use it — stopping it stops the server.


In [ ]:
#@title Start
import os
import pathlib
import subprocess

ROOT = pathlib.Path("/content/ShiroRVC")
python = ROOT / "env" / "bin" / "python"

if not python.exists():
    raise SystemExit("Run the Setup cell first.")

os.chdir(ROOT)

# --share is what makes this reachable: the server binds inside the Colab
# container, so the public link is the only way in. Output is streamed rather
# than captured so the link appears while the server is still starting.
process = subprocess.Popen(
    [str(python), "app.py", "--share"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    process.terminate()
    print()
    print("Stopped.")
